In [ ]:
%%capture
!pip install -q "transformers>=4.48.0" trl==0.11.4 peft==0.13.2 "accelerate>=1.1.0" sentencepiece datasets
!pip install -q "bitsandbytes>=0.46.1" --upgrade
!pip install -q sentence-transformers faiss-cpu rouge-score anthropic

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json, torch, faiss, numpy as np, math
from sentence_transformers import SentenceTransformer

with open('/content/drive/MyDrive/LegalRAG/data/mevzuat_chunked0v2_normalized.json', 'r') as f:
    chunks = json.load(f)

with open('/content/drive/MyDrive/LegalRAG/data/gold_test_normalized_matched_161.json', 'r') as f:
    gold_161 = json.load(f)

chunk_id_to_idx = {c['metadata']['chunk_id']: i for i, c in enumerate(chunks)}
chunk_texts = [c['text'] for c in chunks]

print(f"Chunk: {len(chunks)}, Gold test: {len(gold_161)}")

In [ ]:
# Mursit yükle ve FAISS kur
print("Mursit yükleniyor...")
embedding_model = SentenceTransformer('/content/drive/MyDrive/LegalRAG/models/mursit_large_tur2_v2')

print("Chunk'lar encode ediliyor...")
chunk_emb = embedding_model.encode(chunk_texts, batch_size=64, normalize_embeddings=True, show_progress_bar=True)

index = faiss.IndexFlatIP(chunk_emb.shape[1])
index.add(chunk_emb)
print(f"FAISS hazır: {index.ntotal} vektör")
print(f"GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# BGE Reranker yükle
from transformers import AutoModelForSequenceClassification, AutoTokenizer

print("BGE Reranker yükleniyor...")
reranker_tokenizer = AutoTokenizer.from_pretrained('/content/drive/MyDrive/LegalRAG/models/bge_reranker_ft')
reranker_model = AutoModelForSequenceClassification.from_pretrained('/content/drive/MyDrive/LegalRAG/models/bge_reranker_ft')
reranker_model.eval()
reranker_model.to('cuda')
print(f"GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
!pip install 'accelerate>=1.1.0' -q

In [ ]:
# LLM yükle
from transformers import AutoTokenizer as LLMTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

print("LLM yükleniyor...")

MODEL_ID = 'Trendyol/Trendyol-LLM-8b-chat-v2.0'
ADAPTER_PATH = '/content/drive/MyDrive/LegalRAG/models/trendyol_ft_context'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
)

llm_tokenizer = LLMTokenizer.from_pretrained(ADAPTER_PATH)
llm_tokenizer.pad_token = llm_tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)

llm_model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
llm_model.eval()

print(f"GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print("✅ LLM hazır")

In [ ]:
import re

SYSTEM_PROMPT = (
    "Sen bir Türk hukuku uzmanı asistanısın. "
    "Cevaplarını SADECE verilen kanun metnine dayandır. "
    "Kanun metninde olmayan bilgileri ekleme. "
    "Her cevabın sonunda ilgili kanun maddesini belirt."
)

def rerank(soru, chunk_listesi, batch_size=16):
    skorlar = []
    for i in range(0, len(chunk_listesi), batch_size):
        batch = chunk_listesi[i:i+batch_size]
        encoding = reranker_tokenizer(
            [soru] * len(batch), batch,
            truncation=True, max_length=512,
            padding=True, return_tensors='pt'
        ).to('cuda')
        with torch.no_grad():
            outputs = reranker_model(**encoding)
            batch_skorlar = outputs.logits.squeeze(-1).cpu().tolist()
            if isinstance(batch_skorlar, float):
                batch_skorlar = [batch_skorlar]
            skorlar.extend(batch_skorlar)
    return skorlar

def generate_answer(soru, context, max_new_tokens=256):
    user_content = f"Aşağıdaki kanun maddesine dayanarak soruyu yanıtla:\n\n{context}\n\nSoru: {soru}"
    prompt = (
        f"<|begin_of_text|>"
        f"<|start_header_id|>system<|end_header_id|>\n{SYSTEM_PROMPT}<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n{user_content}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n"
    )
    inputs = llm_tokenizer(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        output = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            repetition_penalty=1.1,
            eos_token_id=llm_tokenizer.eos_token_id,
        )
    generated = output[0][inputs['input_ids'].shape[1]:]
    return llm_tokenizer.decode(generated, skip_special_tokens=True)

def full_pipeline(soru, top_k=20):
    # 1. Dense retrieval
    soru_emb = embedding_model.encode([soru], normalize_embeddings=True)
    _, indices = index.search(soru_emb, top_k)
    top_k_idxler = indices[0].tolist()
    top_k_chunks = [chunk_texts[idx] for idx in top_k_idxler]

    # 2. Rerank
    skorlar = rerank(soru, top_k_chunks)
    sirali = sorted(zip(top_k_idxler, top_k_chunks, skorlar), key=lambda x: x[2], reverse=True)
    top3_chunks = [c for _, c, _ in sirali[:3]]

    # 3. Context birleştir
    context = "\n\n---\n\n".join(top3_chunks)

    # 4. LLM cevap üret
    cevap = generate_answer(soru, context)

    return cevap, context

# Test et
print("=== FULL SYSTEM TEST ===")
test_sorular = [
    "Hırsızlık suçunun cezası nedir?",
    "Kira sözleşmesi nasıl feshedilir?",
]

for soru in test_sorular:
    print(f"\nSoru: {soru}")
    cevap, context = full_pipeline(soru)
    print(f"Cevap: {cevap[:300]}")

In [ ]:
from tqdm import tqdm

print("=== FAİTHFULNESS TESTİ — 161 SORU ===")

sonuclar = []

for soru_data in tqdm(gold_161):
    soru = soru_data['question']
    gold_cevap = soru_data['gold_answer']

    try:
        cevap, context = full_pipeline(soru)
        sonuclar.append({
            "soru": soru,
            "cevap": cevap,
            "context": context,
            "gold_cevap": gold_cevap
        })
    except Exception as e:
        print(f"Hata: {e}")
        sonuclar.append({
            "soru": soru,
            "cevap": "",
            "context": "",
            "gold_cevap": gold_cevap
        })

print(f"\n✅ {len(sonuclar)} soru tamamlandı")
print("\n--- Örnek ---")
print(f"Soru: {sonuclar[0]['soru']}")
print(f"Cevap: {sonuclar[0]['cevap'][:200]}")

In [ ]:
!pip install ragas langchain-anthropic -q

In [ ]:
import anthropic
import json
from tqdm import tqdm

client = anthropic.Anthropic(api_key="API_KEY_BURAYA")

def faithfulness_judge(soru, cevap, context):
    prompt = f"""Aşağıdaki cevabın verilen context'e ne kadar dayandığını 0-1 arasında puanla.

Context:
{context[:1000]}

Soru: {soru}
Cevap: {cevap}

Sadece şu JSON formatında yanıt ver, başka hiçbir şey yazma:
{{"skor": 0.85, "gerekce": "kısa açıklama"}}

Puanlama kriteri:
- 1.0: Cevap tamamen context'e dayanıyor
- 0.5: Cevap kısmen context'e dayanıyor, kısmen kendi bilgisinden
- 0.0: Cevap context'i tamamen görmezden geliyor"""

    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=100,
        messages=[{"role": "user", "content": prompt}]
    )

    try:
        result = json.loads(response.content[0].text)
        return result['skor']
    except:
        return 0.5

print("Faithfulness ölçülüyor (161 soru)...")
skorlar = []
for s in tqdm(sonuclar):
    skor = faithfulness_judge(s['soru'], s['cevap'], s['context'])
    skorlar.append(skor)

print(f"\nFaithfulness: {sum(skorlar)/len(skorlar):.4f}")
print(f"Arkadaşın skoru: ~0.25-0.29")

In [ ]:
from rouge_score import rouge_scorer
import re

def metrik_hesapla(sonuclar):
    from rouge_score import rouge_scorer as rs
    scorer = rs.RougeScorer(['rougeL'], use_stemmer=False)

    f1_skorlar = []
    rouge_skorlar = []
    citation_skorlar = []

    for s in sonuclar:
        cevap = s['cevap'].lower()
        gold = s['gold_cevap'].lower()

        # Token bazlı F1
        cevap_token = set(cevap.split())
        gold_token = set(gold.split())
        if not cevap_token or not gold_token:
            continue

        precision = len(cevap_token & gold_token) / len(cevap_token)
        recall    = len(cevap_token & gold_token) / len(gold_token)
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        f1_skorlar.append(f1)

        # ROUGE-L
        rouge = scorer.score(gold, cevap)
        rouge_skorlar.append(rouge['rougeL'].fmeasure)

        # Citation Accuracy — madde numarası referans ediyor mu?
        gold_maddeler = set(re.findall(r'madde\s*\d+', s['gold_cevap'].lower()))
        cevap_maddeler = set(re.findall(r'madde\s*\d+', cevap))
        if gold_maddeler:
            citation = len(gold_maddeler & cevap_maddeler) / len(gold_maddeler)
        else:
            citation = 1.0
        citation_skorlar.append(citation)

    return {
        "F1":              round(sum(f1_skorlar)/len(f1_skorlar), 4),
        "ROUGE-L":         round(sum(rouge_skorlar)/len(rouge_skorlar), 4),
        "CitationAccuracy":round(sum(citation_skorlar)/len(citation_skorlar), 4),
    }

!pip install rouge-score -q

metrикler = metrik_hesapla(sonuclar)
for m, v in metrикler.items():
    print(f"{m}: {v}")

In [ ]:
from transformers import AutoTokenizer as LLMTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

MODEL_ID = 'Trendyol/Trendyol-LLM-8b-chat-v2.0'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print("Ham Trendyol yükleniyor...")
llm_tokenizer = LLMTokenizer.from_pretrained(MODEL_ID)
llm_tokenizer.pad_token = llm_tokenizer.eos_token

ham_llm = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)
ham_llm.eval()
print(f"GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print("✅ Ham LLM hazır")

In [ ]:
import json, faiss, numpy as np, math
from sentence_transformers import SentenceTransformer

# Ham Mursit yükle
print("Ham Mursit yükleniyor...")
ham_embedding = SentenceTransformer('newmindai/Mursit-Large-TR-Retrieval')

with open('/content/drive/MyDrive/LegalRAG/data/mevzuat_chunked0v2_normalized.json', 'r') as f:
    chunks = json.load(f)

with open('/content/drive/MyDrive/LegalRAG/data/gold_test_normalized_matched_161.json', 'r') as f:
    gold_161 = json.load(f)

chunk_id_to_idx = {c['metadata']['chunk_id']: i for i, c in enumerate(chunks)}
chunk_texts = [c['text'] for c in chunks]

print("Chunk'lar encode ediliyor...")
ham_chunk_emb = ham_embedding.encode(chunk_texts, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
ham_index = faiss.IndexFlatIP(ham_chunk_emb.shape[1])
ham_index.add(ham_chunk_emb)

print(f"✅ Ham FAISS hazır: {ham_index.ntotal} vektör")
print(f"GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
from tqdm import tqdm

SYSTEM_PROMPT = (
    "Sen bir Türk hukuku uzmanı asistanısın. "
    "Cevaplarını SADECE verilen kanun metnine dayandır. "
    "Kanun metninde olmayan bilgileri ekleme. "
    "Her cevabın sonunda ilgili kanun maddesini belirt."
)

def generate_answer(soru, context, model, tokenizer, max_new_tokens=256):
    user_content = f"Aşağıdaki kanun maddesine dayanarak soruyu yanıtla:\n\n{context}\n\nSoru: {soru}"
    prompt = (
        f"<|begin_of_text|>"
        f"<|start_header_id|>system<|end_header_id|>\n{SYSTEM_PROMPT}<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n{user_content}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n"
    )
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id,
        )
    generated = output[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

def sistem_test_et(sistem_adi, embedding_model, faiss_index, llm_model, llm_tok, gold_161, reranker_model=None, reranker_tok=None):
    print(f"\n=== {sistem_adi} ===")
    sonuclar = []

    for soru_data in tqdm(gold_161):
        soru = soru_data['question']
        gold_cevap = soru_data['gold_answer']

        # Retrieval
        soru_emb = embedding_model.encode([soru], normalize_embeddings=True)
        _, indices = faiss_index.search(soru_emb, 20)
        top_k_idxler = indices[0].tolist()
        top_k_chunks = [chunk_texts[idx] for idx in top_k_idxler]

        # Reranker varsa
        if reranker_model is not None:
            skorlar = []
            for i in range(0, len(top_k_chunks), 16):
                batch = top_k_chunks[i:i+16]
                enc = reranker_tok(
                    [soru]*len(batch), batch,
                    truncation=True, max_length=512,
                    padding=True, return_tensors='pt'
                ).to('cuda')
                with torch.no_grad():
                    out = reranker_model(**enc)
                    batch_s = out.logits.squeeze(-1).cpu().tolist()
                    if isinstance(batch_s, float): batch_s = [batch_s]
                    skorlar.extend(batch_s)
            sirali = sorted(zip(top_k_idxler, top_k_chunks, skorlar), key=lambda x: x[2], reverse=True)
            top3_chunks = [c for _, c, _ in sirali[:3]]
        else:
            top3_chunks = top_k_chunks[:3]

        context = "\n\n---\n\n".join(top3_chunks)
        cevap = generate_answer(soru, context, llm_model, llm_tok)

        sonuclar.append({
            "soru": soru,
            "cevap": cevap,
            "context": context,
            "gold_cevap": gold_cevap
        })

    return sonuclar

# 1. Sistem — Baseline RAG
sonuclar_1 = sistem_test_et(
    "1_Baseline_RAG",
    ham_embedding, ham_index,
    ham_llm, llm_tokenizer,
    gold_161
)
print(f"✅ 1_Baseline_RAG tamamlandı: {len(sonuclar_1)} soru")

In [ ]:
import gc
torch.cuda.empty_cache()
gc.collect()

# FT Mursit yükle
print("FT Mursit yükleniyor...")
ft_embedding = SentenceTransformer('/content/drive/MyDrive/LegalRAG/models/mursit_large_tur2_v2')

ft_chunk_emb = ft_embedding.encode(chunk_texts, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
ft_index = faiss.IndexFlatIP(ft_chunk_emb.shape[1])
ft_index.add(ft_chunk_emb)
print(f"✅ FT FAISS hazır: {ft_index.ntotal} vektör")

# 2. Sistem — Embedding Tuning
sonuclar_2 = sistem_test_et(
    "2_Embedding_Tuning",
    ft_embedding, ft_index,
    ham_llm, llm_tokenizer,
    gold_161
)
print(f"✅ 2_Embedding_Tuning tamamlandı: {len(sonuclar_2)} soru")

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer as RerankTokenizer

print("FT BGE Reranker yükleniyor...")
reranker_tokenizer = RerankTokenizer.from_pretrained('/content/drive/MyDrive/LegalRAG/models/bge_reranker_ft')
reranker_model = AutoModelForSequenceClassification.from_pretrained('/content/drive/MyDrive/LegalRAG/models/bge_reranker_ft')
reranker_model.eval()
reranker_model.to('cuda')
print(f"GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB")

# 3. Sistem — Reranker
sonuclar_3 = sistem_test_et(
    "3_Reranker",
    ft_embedding, ft_index,
    ham_llm, llm_tokenizer,
    gold_161,
    reranker_model=reranker_model,
    reranker_tok=reranker_tokenizer
)
print(f"✅ 3_Reranker tamamlandı: {len(sonuclar_3)} soru")

In [ ]:
from peft import PeftModel

print("FT LLM yükleniyor...")
ADAPTER_PATH = '/content/drive/MyDrive/LegalRAG/models/trendyol_ft_context'

ft_llm = PeftModel.from_pretrained(ham_llm, ADAPTER_PATH)
ft_llm.eval()
print(f"GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB")

# 4. Sistem — LLM Fine-tuning
sonuclar_4 = sistem_test_et(
    "4_LLM_Finetuning",
    ft_embedding, ft_index,
    ft_llm, llm_tokenizer,
    gold_161,
    reranker_model=reranker_model,
    reranker_tok=reranker_tokenizer
)
print(f"✅ 4_LLM_Finetuning tamamlandı: {len(sonuclar_4)} soru")

# 5. Full System — 4 ile aynı
sonuclar_5 = sonuclar_4.copy()
print(f"✅ 5_Full_System tamamlandı: {len(sonuclar_5)} soru")

In [ ]:
!pip install rouge-score -q

from rouge_score import rouge_scorer as rs
import re, anthropic, json
from tqdm import tqdm

scorer = rs.RougeScorer(['rougeL'], use_stemmer=False)

def metrikleri_hesapla(sonuclar, sistem_adi):
    f1_skorlar, rouge_skorlar, citation_skorlar = [], [], []

    for s in sonuclar:
        cevap = s['cevap'].lower()
        gold = s['gold_cevap'].lower()

        # F1
        cevap_token = set(cevap.split())
        gold_token = set(gold.split())
        if cevap_token and gold_token:
            precision = len(cevap_token & gold_token) / len(cevap_token)
            recall    = len(cevap_token & gold_token) / len(gold_token)
            f1 = 2*precision*recall/(precision+recall) if (precision+recall) > 0 else 0
            f1_skorlar.append(f1)

        # ROUGE-L
        rouge = scorer.score(gold, cevap)
        rouge_skorlar.append(rouge['rougeL'].fmeasure)

        # Citation Accuracy
        gold_maddeler = set(re.findall(r'madde\s*\d+', s['gold_cevap'].lower()))
        cevap_maddeler = set(re.findall(r'madde\s*\d+', cevap))
        if gold_maddeler:
            citation_skorlar.append(len(gold_maddeler & cevap_maddeler) / len(gold_maddeler))
        else:
            citation_skorlar.append(1.0)

    return {
        "Sistem": sistem_adi,
        "N": len(sonuclar),
        "F1":               round(sum(f1_skorlar)/len(f1_skorlar), 4),
        "ROUGE-L":          round(sum(rouge_skorlar)/len(rouge_skorlar), 4),
        "CitationAccuracy": round(sum(citation_skorlar)/len(citation_skorlar), 4),
    }

# Tüm sistemler için hesapla
sistemler = [
    (sonuclar_1, "1_Baseline_RAG"),
    (sonuclar_2, "2_Embedding_Tuning"),
    (sonuclar_3, "3_Reranker"),
    (sonuclar_4, "4_LLM_Finetuning"),
    (sonuclar_5, "5_Full_System"),
]

metrik_sonuclari = []
for sonuclar, isim in sistemler:
    m = metrikleri_hesapla(sonuclar, isim)
    metrik_sonuclari.append(m)
    print(f"{isim}: F1={m['F1']}, ROUGE-L={m['ROUGE-L']}, Citation={m['CitationAccuracy']}")

In [ ]:
client = anthropic.Anthropic(api_key="API_KEY_BURAYA")

def faithfulness_judge(soru, cevap, context):
    prompt = f"""Aşağıdaki cevabın verilen context'e ne kadar dayandığını 0-1 arasında puanla.

Context:
{context[:1000]}

Soru: {soru}
Cevap: {cevap}

Sadece şu JSON formatında yanıt ver, başka hiçbir şey yazma:
{{"skor": 0.85, "gerekce": "kısa açıklama"}}

Puanlama:
- 1.0: Tamamen context'e dayanıyor
- 0.5: Kısmen context'e dayanıyor
- 0.0: Context'i görmezden geliyor"""

    response = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=100,
        messages=[{"role": "user", "content": prompt}]
    )
    try:
        return json.loads(response.content[0].text)['skor']
    except:
        return 0.5

# Her sistem için faithfulness ölç
for m, (sonuclar, isim) in zip(metrik_sonuclari, sistemler):
    print(f"\n{isim} faithfulness ölçülüyor...")
    skorlar = []
    for s in tqdm(sonuclar[:161]):
        skor = faithfulness_judge(s['soru'], s['cevap'], s['context'])
        skorlar.append(skor)
    m['Faithfulness'] = round(sum(skorlar)/len(skorlar), 4)
    print(f"Faithfulness: {m['Faithfulness']}")

In [ ]:
def answer_quality_judge(soru, cevap, gold_cevap):
    prompt = f"""Aşağıdaki model cevabının kalitesini gold cevapla karşılaştırarak 0-1 arasında puanla.

Soru: {soru}
Gold cevap: {gold_cevap}
Model cevabı: {cevap}

Sadece şu JSON formatında yanıt ver, başka hiçbir şey yazma:
{{"skor": 0.85, "gerekce": "kısa açıklama"}}

Puanlama:
- 1.0: Model cevabı gold cevapla tamamen örtüşüyor
- 0.5: Kısmen doğru, eksik veya fazla bilgi var
- 0.0: Yanlış veya alakasız cevap"""

    response = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=100,
        messages=[{"role": "user", "content": prompt}]
    )
    try:
        return json.loads(response.content[0].text)['skor']
    except:
        return 0.5

# Tüm sistemler için answer quality ölç
for m, (sonuclar, isim) in zip(metrik_sonuclari, sistemler):
    print(f"\n{isim} answer quality ölçülüyor...")
    skorlar = []
    for s in tqdm(sonuclar):
        skor = answer_quality_judge(s['soru'], s['cevap'], s['gold_cevap'])
        skorlar.append(skor)
    m['AnswerQuality'] = round(sum(skorlar)/len(skorlar), 4)
    print(f"AnswerQuality: {m['AnswerQuality']}")

In [ ]:
import pandas as pd

# Retrieval metrikleri elimizde var
retrieval = {
    "1_Baseline_RAG":     {"Recall@5": 0.7341, "Recall@10": 0.8206, "MRR": 0.5752, "nDCG@10": 0.6345},
    "2_Embedding_Tuning": {"Recall@5": 0.8868, "Recall@10": 0.9329, "MRR": 0.7575, "nDCG@10": 0.8005},
    "3_Reranker":         {"Recall@5": 0.9281, "Recall@10": 0.9479, "MRR": 0.8457, "nDCG@10": 0.8724},
    "4_LLM_Finetuning":   {"Recall@5": 0.9281, "Recall@10": 0.9479, "MRR": 0.8457, "nDCG@10": 0.8724},
    "5_Full_System":      {"Recall@5": 0.9281, "Recall@10": 0.9479, "MRR": 0.8457, "nDCG@10": 0.8724},
}

# Faithfulness ve answer quality ekle
faithfulness_scores = {
    "1_Baseline_RAG":     0.7519,
    "2_Embedding_Tuning": 0.71,
    "3_Reranker":         0.7702,
    "4_LLM_Finetuning":   0.7739,
    "5_Full_System":      0.7581,
}

answer_quality_scores = {
    "1_Baseline_RAG":     0.5811,
    "2_Embedding_Tuning": 0.5984,
    "3_Reranker":         0.5969,
    "4_LLM_Finetuning":   0.5820,
    "5_Full_System":      0.5820,
}

f1_scores = {
    "1_Baseline_RAG":     0.4389,
    "2_Embedding_Tuning": 0.4411,
    "3_Reranker":         0.4398,
    "4_LLM_Finetuning":   0.4391,
    "5_Full_System":      0.4391,
}

rouge_scores = {
    "1_Baseline_RAG":     0.5258,
    "2_Embedding_Tuning": 0.5361,
    "3_Reranker":         0.5327,
    "4_LLM_Finetuning":   0.5231,
    "5_Full_System":      0.5231,
}

# Tablo 1 — Ablation
rows = []
for sistem in ["1_Baseline_RAG", "2_Embedding_Tuning", "3_Reranker", "4_LLM_Finetuning", "5_Full_System"]:
    r = retrieval[sistem]
    rows.append({
        "System":            sistem,
        "N":                 161,
        # Retrieval
        "Recall@5":          r["Recall@5"],
        "Recall@10":         r["Recall@10"],
        "MRR":               r["MRR"],
        "nDCG@10":           r["nDCG@10"],
        # Answer Quality
        "F1":                f1_scores[sistem],
        "ROUGE-L":           rouge_scores[sistem],
        "AnswerQuality(LLM)":answer_quality_scores[sistem],
        # Grounding
        "Faithfulness":      faithfulness_scores[sistem],
        "Hallucination":     round(1 - faithfulness_scores[sistem], 4),
    })

df_ablation = pd.DataFrame(rows)
print("=== TABLO 1 — ABLATION ===")
print(df_ablation.to_string(index=False))

# Tablo 2 — Baseline vs Full System
df_comparison = df_ablation[df_ablation["System"].isin(["1_Baseline_RAG", "5_Full_System"])].copy()
print("\n=== TABLO 2 — BASELINE vs FULL SYSTEM ===")
print(df_comparison.to_string(index=False))

# Drive'a kaydet
df_ablation.to_csv('/content/drive/MyDrive/LegalRAG/outputs/tables/ablation_full.csv', index=False)
df_comparison.to_csv('/content/drive/MyDrive/LegalRAG/outputs/tables/baseline_vs_full.csv', index=False)
print("\n✅ Drive'a kaydedildi")